In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer

from src.data import PROCESSED
from src.dl_dataset import ApneaDataset, load_cache
from src.eval import evaluate_and_log

folds_df = pd.read_csv(PROCESSED / "folds.csv")
cache = load_cache(folds_df["record"].unique())

In [ ]:
ds = ApneaDataset(folds_df, cache, mode="rr_edr", context_mins=3, augment=False)
X = np.stack([ds[i][0].numpy() for i in range(len(ds))])  # (N, 2, 540)
y = folds_df["label"].to_numpy()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
net = MantisV2(device=device).from_pretrained("paris-noah/MantisV2")
fm = MantisTrainer(device=device, network=net)
Z = fm.transform(X)

In [ ]:
oof_probs = np.zeros(len(folds_df))
for f in range(5):
    tr = folds_df["patient_fold"] != f
    va = folds_df["patient_fold"] == f
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.5))
    clf.fit(Z[tr], y[tr])
    oof_probs[va] = clf.predict_proba(Z[va])[:, 1]

evaluate_and_log(
    folds_df, oof_probs,
    model_name="mantisv2_zs_lr_rredr_3m",
    split_type="patient_fold",
    config={"backbone": "paris-noah/MantisV2", "mode": "rr_edr", "context_mins": 3, "head": "lr"},
    use_wandb=True,
)